In [0]:
##Query to find employees with equal salaries

print("-----GIVEN DATAFRAME-----\n\n\n")

import datetime 

data = [
    ("001", "Monika",   "Arora",   100000, datetime.datetime(2014, 2, 20, 9, 0, 0), "HR"),
    ("002", "Niharika", "Verma",   300000, datetime.datetime(2014, 6, 11, 9, 0, 0), "Admin"),
    ("003", "Vishal",   "Singhal", 300000, datetime.datetime(2014, 2, 20, 9, 0, 0), "HR"),
    ("004", "Amitabh",  "Singh",   500000, datetime.datetime(2014, 2, 20, 9, 0, 0), "Admin"),
    ("005", "Vivek",    "Bhati",   500000, datetime.datetime(2014, 6, 11, 9, 0, 0), "Admin"),
]

columns = ["workerid", "firstname", "lastname", "salary", "joiningdate", "depart"]

df = spark.createDataFrame(data, columns)
df.show()


print("-----REQUIRED DATAFRAME-----\n\n\n")

from pyspark.sql.functions import to_timestamp

data1 = [
    ("002", "Niharika", "Verma",   300000, "2014-06-11 09:00:00", "Admin"),
    ("003", "Vishal",   "Singhal", 300000, "2014-02-20 09:00:00", "HR"),
    ("004", "Amitabh",  "Singh",   500000, "2014-02-20 09:00:00", "Admin"),
    ("005", "Vivek",    "Bhati",   500000, "2014-06-11 09:00:00", "Admin"),
]

columns1 = ["workerid", "firstname", "lastname", "salary", "joiningdate", "depart"]

df1 = spark.createDataFrame(data1, columns1)
df1 = df1.withColumn("joiningdate", to_timestamp("joiningdate", "yyyy-MM-dd HH:mm:ss"))

df1.show()
df1.printSchema()

print("-----DSL SOLUTION-----\n\n\n")

from pyspark.sql.functions import expr,col


required_df = (
    df.withColumn("salary_count", expr("count(salary) over (partition by salary)"))
      .filter(col("salary_count") > 1)
      .orderBy("salary")
      .drop("salary_count")
)

required_df.show()



print("-----SQL SOLUTION-----\n\n\n")


df.createOrReplaceTempView("df_view")

result = spark.sql("""
    SELECT * FROM (
        SELECT *,
               COUNT(salary) OVER (PARTITION BY salary) AS salary_count
        FROM df_view
    )
    WHERE salary_count > 1
    ORDER BY salary
""").drop("salary_count")

result.show()




